### Validamos base

In [1]:

import sys 
from sqlalchemy import create_engine, text

import numpy as np
import sys 
sys.path.append('C:/Users/Data/Documents/lazo_fernando/target_script_01/funciones')
from funciones import *
from variables_inicio import *
from funciones_spark import *
from utils_sql import *

spark = SparkSession.builder \
    .appName("SparkExample") \
    .master("local[*]") \
    .config('spark.driver.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.4.0.jre11.jar') \
    .config('spark.executor.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.4.0.jre11.jar') \
    .config('spark.executor.memory', '8g') \
    .config('spark.driver.memory', '8g') \
    .getOrCreate()


server_sql = server_kishin
db_sql = "DANTALION"
user_sql = user_kishin
pwd_sql = pwd_kishin

engine_kishin = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)
server_sql = server_zeus
db_sql = "odin"
user_sql = user_zeus
pwd_sql = pwd_zeus

engine_odin = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)

server_sql = server_zeus
db_sql = "SAMANTHA"
user_sql = user_zeus
pwd_sql = pwd_zeus

engine_samantha = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)


engine_mysql = create_engine(
    f"mysql+pymysql://{user_envio}:{pwd_envio}@{server_envio}:{port_mysql}/{db_envio}"
)



fecha_mes_base='2026-08-01'

c:\Users\Data\Documents\lazo_fernando\target_script_01\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [2]:

query = f"""
	SELECT dni_cliente as Dni,'1' as ref
    FROM Alice.prospectos_correos_alfin 
    where fecha_registro>='2026-08-01'
"""
df_correo_ref = pd.read_sql(query, engine_mysql)

ruta_archivo = os.path.join(ruta_alfin, 'envio_subir.csv')
df_correo_ref.to_csv(ruta_archivo, sep=',')

filename='envio_subir.csv'

df_correo_01=cargar_archivo_csv_ruta(spark,filename,',',True,ruta_alfin)

In [8]:
query = """
    select *,NUMERO_DOCUMENTO as Dni   
    from DANTALION.dbo.Base_Maestra_ALFIN_BK
    where cl_telf1 is not null
    and fecha_envio>='2026-08-01'
    AND RETIRO = 'ACTIVO'
    and cruce='CET'
    """
df_formato=obtener_tabla_sql(spark,query,server_kishin,user_kishin,pwd_kishin,db_kishin)

query = """
	SELECT distinct Dni FROM THOTH.dbo.Tmp_LLamadas_Alfin_5 
    where Descripcion_ in(
        'EXPRESO FUTURA DENUNCIA ANTE INDECOPI O REGULADOR',
        'EXPRESO QUE NO AUTORIZÓ USO DE DATOS PERSONALES',
        'EXPRESO RECIBIR MÚLTIPLES LLAMADAS',
        'FUERA DE SERVICIO'
    )
    """
df_quitar=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)

query = """
    SELECT Dni FROM THOTH.dbo.Tmp_LLamadas_Alfin_5 
    where Numero_Campana='401'
    and list_description<>'provicional'
    """
df_quitar2=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)

print(df_formato.count())
print(df_quitar.count())
print(df_quitar2.count())

23174
1479
664


In [9]:
df_formato=df_formato.join(df_quitar, ['Dni'], "left_anti")
df_formato=df_formato.join(df_quitar2, ['Dni'], "left_anti")

In [10]:
print(df_formato.count())

df_formato=df_formato.join(df_correo_01, ['Dni'], "left_anti")
print(df_formato.count())




23039
8439


In [ ]:
['Dni', 'TIPO_DOI', 'NUMERO_DOCUMENTO', 'NOMBRES', 'APELLIDO_PATERNO', 'APELLIDO_MATERNO', 'SUCURSAL', 'TIENDA', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'FEC_NACIMIENTO', 'OFERTA_MAX', 'OFERTA_REEN', 'Tipo_verificacion', 'GRUPO_RIESGO', 'proveedor', 'lote', 'RETIRO', 'Tasa_1', 'Tasa_2', 'Tasa_3', 'Tasa_4', 'Tasa_5', 'Tasa_6', 'Tasa_7', 'segmento', 'Campana', 'PLAZO', 'TEM', 'PROPENSION_IC', 'Desgravamen', 'CUOTA', 'Edad', 'Oferta_12M', 'Tasa_12M', 'Desgravamen_12M', 'CUOTA_12M', 'Oferta_18M', 'Tasa_18M', 'Desgravamen_18M', 'CUOTA_18M', 'Oferta_24M', 'Tasa_24M', 'Desgravamen_24M', 'CUOTA_24M', 'Oferta_36M', 'Tasa_36M', 'Desgravamen_36M', 'CUOTA_36M', 'Validador_Telefono', 'Prioridad', 'Nombre_prioridad', 'Deuda_1', 'Entidad_1', 'Deuda_2', 'Entidad_2', 'Deuda_3', 'Entidad_3', 'sucursal_comercial', 'Agencia_comercial', 'Region_comercial', 'Ubicacion', 'OfertaMaximaSinSeguro', 'color', 'color_final', 'PROPENSION', 'OFERTA_FINAL', 'GARANTIA', 'Oferta_Minima_Paperless', 'RANGO_OFERTA', 'RANGO_SUELDO', 'CAPACIDAD_MAX', 'PEER', 'PROP_COMER', 'TIPO_GEST', 'CLIENTE_NUEVO', 'GRUPO_TASA', 'NUEVOS_3M', 'NUEVOS_6M', 'NUEVOS_9M', 'NUEVOS_12M', 'NUEVOS_4M', 'GRUPO_MONTO', 'TASA_VS_MONTO', 'USUARIO', 'incremento_monto_riesgos', 'FLG_DEUDA_PLUS', 'tipo_cliente_riegos', 'USER_V3', 'LEAD_CALIDAD', 'SEGMENTO_USER', 'RANGO_EDAD', 'RANGO_OFERTA2', 'PERIODO', 'RETIRO_GEST', 'MEJOR_TIPIFICACION', 'STATUS', 'FECHA_SOL', 'BASE', 'RESULTADO', 'NUM_ENRIQUECIDO', 'TIPO_CONTACTO', 'Q_VENTAS', 'LOCALIDAD', 'DESEMBOLSADO', 'MONTO_DESEMBOLSADO', 'SBI', 'CRUCE', 'PREST_PREVIO', 'ID_CLIENTE', 'RANGO_EDAD2', 'Fecha_Envio', 'TIPO_BD', 'COD_BD', 'NOMB_BD', 'MES_GESTION', 'TIPO_CLIENTE', 'GRUPO_TASA_REENGANCHE', 'SALDO_DIFERENCIAL_REENG', 'FLAG_REENG', 'RETIRO_DESEMBOLSO', 'FRESCURA', 'flag_deuda_v_oferta', 'MGNEG', 'PERFIL_RO', 'TIPO_BASE', 'cl_telf1', 'cl_telf2', 'cl_telf3', 'cl_telf4', 'cl_telf5', 'cl_telf6', 'cl_telf7', 'cl_telf8', 'cl_telf9', 'cl_telf10', 'cl_movil', 'cl_celular', 'cl_telefono', 'cl_turno', 'cl_gestor', 'cl_asesor', 'cl_accion', 'cl_gestion', 'SERVICIO', 'cl_fecha_gestion', 'cl_hora_gestion', 'cl_hits', 'cl_fecha_llamar', 'cl_prioridad', 'cl_orden', 'cl_predictivo', 'cl_tiempo', 'cl_base', 'cl_mes', 'cl_carga', 'id_carga', 'cl_area', 'fecha_alimentacion', 'cl_base_ant', 'cl_accion_ant', 'cl_fecha_ant', 'campania', 'PROMOCION', 'PROMOCION2', 'nombre_base', 'NumEntidades', 'p_banco', 'PERFIL_GLOBAL', 'FLG_AAHH', 'SCORE_TELEFONO', 'PILOTO_PLAZAS', 'INTENSIDAD_MAX', 'marca1', 'marca2', 'marca3', 'AÑO_DURACION_BASE', 'MES_DURACION_BASE', 'FLAT2', 'REP1', 'REP2', 'PILOTO_RETENCION', 'CAMP_BONO', 'ACCION']colo

['Dni', 'TIPO_DOI', 'NUMERO_DOCUMENTO', 'NOMBRES', 'APELLIDO_PATERNO', 'APELLIDO_MATERNO', 'SUCURSAL', 'TIENDA', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'FEC_NACIMIENTO', 'OFERTA_MAX', 'OFERTA_REEN', 'Tipo_verificacion', 'GRUPO_RIESGO', 'proveedor', 'lote', 'RETIRO', 'Tasa_1', 'Tasa_2', 'Tasa_3', 'Tasa_4', 'Tasa_5', 'Tasa_6', 'Tasa_7', 'segmento', 'Campana', 'PLAZO', 'TEM', 'PROPENSION_IC', 'Desgravamen', 'CUOTA', 'Edad', 'Oferta_12M', 'Tasa_12M', 'Desgravamen_12M', 'CUOTA_12M', 'Oferta_18M', 'Tasa_18M', 'Desgravamen_18M', 'CUOTA_18M', 'Oferta_24M', 'Tasa_24M', 'Desgravamen_24M', 'CUOTA_24M', 'Oferta_36M', 'Tasa_36M', 'Desgravamen_36M', 'CUOTA_36M', 'Validador_Telefono', 'Prioridad', 'Nombre_prioridad', 'Deuda_1', 'Entidad_1', 'Deuda_2', 'Entidad_2', 'Deuda_3', 'Entidad_3', 'sucursal_comercial', 'Agencia_comercial', 'Region_comercial', 'Ubicacion', 'OfertaMaximaSinSeguro', 'color', 'color_final', 'PROPENSION', 'OFERTA_FINAL', 'GARANTIA', 'Oferta_Minima_Paperless', 'RANGO_OFERTA', 'RAN

In [11]:
df_formato=df_formato.filter(F.col('OFERTA_MAX')>=5000)


In [14]:
df_formato=df_formato.filter(~F.col('USER_V3').isin('7. Peers','4. MES + PLD No Peers','2. sunedu & sunarp B','1. sunedu & sunarp A'))


In [16]:
df_formato.groupBy('color_final') \
    .count() \
    .orderBy('color_final') \
    .show(30)

+---------------+-----+
|    color_final|count|
+---------------+-----+
| AMARILLO CLARO|  856|
|AMARILLO OSCURO| 1568|
|  NARANJA CLARO|  511|
| NARANJA OSCURO|  460|
|    VERDE CLARO|  343|
|   VERDE OSCURO|  612|
+---------------+-----+



In [ ]:
df_formato=df_formato.filter(F.col('OFERTA_MAX')>=5000)


In [20]:
df_formato=df_formato.filter(F.col('color_final').isin('VERDE OSCURO','VERDE CLARO','','AMARILLO OSCURO'))

In [ ]:
df_formato=df_formato.filter(~F.col('USER_V3').isin('7. Peers','4. MES + PLD No Peers','2. sunedu & sunarp B','1. sunedu & sunarp A'))


In [4]:
df_f=cargar_archivo_csv_ruta(spark,'validar_campana_alfin_bloqueo.csv',';',True,ruta_alfin)

In [6]:
print(df_f.columns)

['DNI', '_c0', 'CLIENTE', 'CELULAR', 'AGENCIA', 'MONTO', 'COLOR_FINAL', 'COD_USER_V3', 'USER_V3', 'PERFIL_RO', 'campaña', 'OFERTA_MAX', 'PLAZO', 'CAPACIDAD_MAX', 'FRESCURA', 'rango_deuda', 'numentidades', 'TOTAL_A_LIQUIDAR', 'TASA_CREDITO_ANTERIOR', 'TASA_1', 'TASA_2', 'TASA_3', 'TASA_4', 'TASA_5', 'TASA_6', 'TASA_7', 'MGNEG', 'MARCA_PD', 'AUTORIZACION_DATOS', 'FLAG_DEUDA_V_OFERTA', 'GRUPO_TASA', 'TIPO_BASE', 'PROPENSION_DISTRIBUCION', 'OFERTA_SS', 'TASA_1_SS', 'TASA_2_SS', 'TASA_3_SS', 'TASA_4_SS', 'TASA_5_SS', 'TASA_6_SS', 'TASA_7_SS', 'ALERTA_MAQUETA', 'FEN', 'PERFIL_ESPECIAL', 'TASA_MIN_DESCUENTO', 'TIPO']


In [ ]:
['DNI', '_c0', 'CLIENTE', 'CELULAR', 'AGENCIA', 'MONTO', 'COLOR_FINAL', 'COD_USER_V3', 'USER_V3', 'PERFIL_RO', 'campaña', 'OFERTA_MAX', 'PLAZO', 'CAPACIDAD_MAX', 'FRESCURA', 'rango_deuda', 'numentidades', 'TOTAL_A_LIQUIDAR', 'TASA_CREDITO_ANTERIOR', 'TASA_1', 'TASA_2', 'TASA_3', 'TASA_4', 'TASA_5', 'TASA_6', 'TASA_7', 'MGNEG', 'MARCA_PD', 'AUTORIZACION_DATOS', 'FLAG_DEUDA_V_OFERTA', 'GRUPO_TASA', 'TIPO_BASE', 'PROPENSION_DISTRIBUCION', 'OFERTA_SS', 'TASA_1_SS', 'TASA_2_SS', 'TASA_3_SS', 'TASA_4_SS', 'TASA_5_SS', 'TASA_6_SS', 'TASA_7_SS', 'ALERTA_MAQUETA', 'FEN', 'PERFIL_ESPECIAL', 'TASA_MIN_DESCUENTO', 'TIPO']ce

In [7]:
df_formato_pd=df_f.select(F.col('DNI').alias('dni_cliente'),
F.col('COLOR_FINAL').alias('color'),
F.col('AGENCIA').alias('agencia_atencion'),
F.col('CELULAR').alias('celular'),
F.col('CELULAR').alias('telefono_cliente'),
F.col('OFERTA_MAX').alias('monto_solicitado'),
F.col('CLIENTE').alias('nombre_cliente')).toPandas()

c:\Users\Data\Documents\lazo_fernando\target_script_01\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Users\Data\Documents\lazo_fernando\target_script_01\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


In [21]:
df_formato_pd=df_formato.select(F.col('NUMERO_DOCUMENTO').alias('dni_cliente'),
F.col('color_final').alias('color'),
F.col('Agencia_comercial').alias('agencia_atencion'),
F.col('cl_telf1').alias('celular'),
F.col('cl_telf1').alias('telefono_cliente'),
F.col('OFERTA_MAX').alias('monto_solicitado'),
F.col('NOMBRES').alias('nombre_cliente')).toPandas()

c:\Users\Data\Documents\lazo_fernando\target_script_01\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Users\Data\Documents\lazo_fernando\target_script_01\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


In [8]:
df_formato_pd['dni_vendedor']='BOT'
df_formato_pd['cdv_alfin_banco']='ROSA HONOR'
df_formato_pd['canal_campo']='CALL CENTER / TARGET OUTSOURCING'
df_formato_pd['codigo_ejecutivo_id']='BOT'
df_formato_pd['ejecutivo_target']='00000001'
df_formato_pd['operador']='TARGET'
df_formato_pd['tipo_gestion']='Derivacion'
df_formato_pd['tipo_carga']='MANUAL'
df_formato_pd['supervisor']='CARLOS ENRIQUE RAMIREZ CACHIQUE'


In [9]:
fechas = pd.to_datetime([
    "2026-08-28",
    "2026-08-31",
    "2026-08-29",
])

df_formato_pd["fecha_visita"] = np.random.choice(
    fechas,
    size=len(df_formato_pd)
)

horas = np.random.randint(9, 19, size=len(df_formato_pd))

minutos = np.random.choice([0, 15, 30, 45], size=len(df_formato_pd))

df_formato_pd["hora_visita"] = [
    f"{h:02d}:{m:02d}:00"
    for h, m in zip(horas, minutos)
]



In [10]:
query = f"""
	select *,agencia_correo as agencia_atencion ,agencia_Formulario as agencia_tienda from Alice.agencias_alfin
"""
df_agencia = pd.read_sql(query, engine_mysql)
df_agencia.drop_duplicates(subset=["agencia_atencion"], inplace=True)

df_agencia["agencia_atencion"] = df_agencia["agencia_atencion"].str.strip()
df_formato_pd["agencia_atencion"] = df_formato_pd["agencia_atencion"].str.strip()



equivalencias = {
    'SAN JUAN DE LURIG': 'SAN JUAN DE LURIGANCHO',
    'ENMANCIPACION': 'EMANCIPACION',
    'PC HUANCAYO': 'HUANCAYO',
    'PC TACNA': 'TACNA',
    'PC HUARAZ': 'HUARAZ',
    'TRUJ CENTRO': 'TRUJILLO CENTRO',
    'TRUJ AMERICA': 'TRUJILLO AMERICA',
    'AREQ CAYMA': 'AREQUIPA CAYMA',
    'AREQ PAMPILLA': 'AREQUIPA PAMPILLA'
}

df_formato_pd['agencia_atencion'] = (
    df_formato_pd['agencia_atencion']
    .replace(equivalencias)
)
# df_formato_pd.drop_duplicates(subset=["dni"], inplace=True)



set_correo = set(
    df_formato_pd['agencia_atencion']
    .dropna()
    .drop_duplicates()
)

set_agencia = set(
    df_agencia['agencia_atencion']
    .dropna()
    .drop_duplicates()
)
# print(set_correo & set_agencia)
print(set_agencia - set_correo)
print(set_correo -set_agencia )


set()
set()


In [11]:
df_formato_pd=df_formato_pd.merge(df_agencia, on='agencia_atencion', how='left')


In [12]:
df_correo=df_formato_pd[['canal_campo', 'supervisor', 'ejecutivo_target', 'codigo_ejecutivo_id', 'cdv_alfin_banco', 'dni_cliente', 'nombre_cliente', 'monto_solicitado', 'celular', 'agencia_atencion', 'fecha_visita','hora_visita','color']] .copy()

df_formulario=df_formato_pd[['dni_vendedor', 'operador', 'dni_cliente', 'nombre_cliente', 'telefono_cliente', 'agencia_tienda', 'fecha_visita', 'monto_solicitado', 'tipo_gestion']].copy()

display(df_correo.head(2))
display(df_formulario.head(2))

,canal_campo,supervisor,ejecutivo_target,codigo_ejecutivo_id,cdv_alfin_banco,dni_cliente,nombre_cliente,monto_solicitado,celular,agencia_atencion,fecha_visita,hora_visita,color
0,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,00000001,BOT,ROSA HONOR,25180454,ISAC HUANCA CHURA,14200,991359730,CUSCO LA CULTURA,2026-08-28,13:30:00,VERDE CLARO
1,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,00000001,BOT,ROSA HONOR,25180454,ISAC HUANCA CHURA,14200,991359730,CUSCO LA CULTURA,2026-08-29,14:00:00,VERDE CLARO


,dni_vendedor,operador,dni_cliente,nombre_cliente,telefono_cliente,agencia_tienda,fecha_visita,monto_solicitado,tipo_gestion
0,BOT,TARGET,25180454,ISAC HUANCA CHURA,991359730,734299 - CUSCO LA CULTURA,2026-08-28,14200,Derivacion
1,BOT,TARGET,25180454,ISAC HUANCA CHURA,991359730,734299 - CUSCO LA CULTURA,2026-08-29,14200,Derivacion


In [13]:
filename='TARGET.txt'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_target_desembolso = pd.read_csv(ruta_archivo,sep='|')
df_target_desembolso = df_target_desembolso[['DNI']].copy()
df_target_desembolso['CANAL']='CANAL'
filename='ACUM_DESEM.txt'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_fugas = pd.read_csv(ruta_archivo,sep='|')
df_fugas = df_fugas[['DNI','CANALVENTA']].copy()
df_target_desembolso['DNI'] = (
    df_target_desembolso['DNI']
    .astype(str)
    .str.replace(r'\D', '', regex=True)   
    .replace('', pd.NA)                     
    .str.zfill(8)                           
)
df_fugas['DNI'] = (
    df_fugas['DNI']
    .astype(str)
    .str.replace(r'\D', '', regex=True)   
    .replace('', pd.NA)                     
    .str.zfill(8)                           
)

df_desembolso=df_fugas.merge(
    df_target_desembolso,
    on=['DNI'],
    how='left'
)
df_desembolso = df_desembolso.fillna("OTROS")
df_desembolso.rename(columns={'DNI': 'dni_cliente'}, inplace=True)


filename='RetiroDefinitivo_BlackList.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_def_blacklist = pd.read_csv(ruta_archivo,sep='|')
filename='RetiroDeGestion_BlackList.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_blacklist = pd.read_csv(ruta_archivo,sep='|')
filename='RetiroDeGestion_Telefonos.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_telf = pd.read_csv(ruta_archivo,sep='|')
filename='retiro_correo_alfin.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_retiro_correo = pd.read_csv(ruta_archivo,sep=',')

# filename='desembolso.csv'
# ruta_archivo = os.path.join(ruta_alfin, filename)
# df_des = pd.read_csv(ruta_archivo,sep=';')

df_def_blacklist = df_def_blacklist.rename(columns={'DNI': 'dni_cliente'})
df_blacklist = df_blacklist.rename(columns={'DNI': 'dni_cliente'})
df_telf= df_telf.rename(columns={'TELEFONO': 'celular'})
df_retiro_correo= df_retiro_correo.rename(columns={'DNI': 'dni_cliente'})

# Blacklists de DNI
# df6 = df_des.copy()
# df6["celular"] = None
# df6 = df6[["dni_cliente", "celular"]]

# Blacklists de DNI
df1 = df_def_blacklist.copy()
df1["celular"] = None
df1 = df1[["dni_cliente", "celular"]]

df2 = df_blacklist.copy()
df2["celular"] = None
df2 = df2[["dni_cliente", "celular"]]

# Blacklist de teléfonos
df3 = df_telf.copy()
df3["dni_cliente"] = None
df3 = df3[["dni_cliente", "celular"]]

# Archivo con DNI y celular
df4 = df_retiro_correo[["dni_cliente", "celular"]].copy()

# Unir todo
df_retiros = pd.concat(
    [df1, df2, df3, df4],
    ignore_index=True
)




dni_retiro = set(df_retiros['dni_cliente'].dropna())
cel_retiro = set(df_retiros['celular'].dropna())
dni_desembolso = set(df_desembolso['dni_cliente'].dropna())


In [14]:
df_formato_pd=df_formato_pd[
    ~df_formato_pd['dni_cliente'].isin(dni_retiro)&
    ~df_formato_pd['celular'].isin(cel_retiro)&
    ~df_formato_pd['dni_cliente'].isin(dni_desembolso)
    ].copy()
df_formato_pd.shape


(3093, 24)

In [15]:

df_correo.to_sql(
    name="prospectos_correos_alfin",
    con=engine_mysql,
    if_exists="append",
    index=False,
    chunksize=1000
)

df_formulario.to_sql(
    name="prospectos_envio_alfin",
    con=engine_mysql,
    if_exists="append",
    index=False,
    chunksize=1000
)

3100